In [2]:
import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

In [4]:
!git clone https://${GITHUB_TOKEN}@github.com/ChinmayeeAwale/dlgenaiproject.git

Cloning into 'dlgenaiproject'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.


In [5]:
%cd dlgenaiproject

/content/dlgenaiproject


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install datasets -q

In [7]:
# Cell 5

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

# Loading Dataset Commonsense_qa from Hugging Face

In [8]:

from datasets import load_dataset
dataset = load_dataset("tau/commonsense_qa")
print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/7.39k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.25MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  160kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1140
    })
})
{'id': '075e483d21c29a511267ef62bedc0461', 'question': 'The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?', 'question_concept': 'punishing', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']}, 'answerKey': 'A'}


# Understanding the Data

In [9]:
# Cell 2
from datasets import load_dataset
dataset = load_dataset("tau/commonsense_qa")
print(dataset)
print(dataset["train"][0])

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1140
    })
})
{'id': '075e483d21c29a511267ef62bedc0461', 'question': 'The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?', 'question_concept': 'punishing', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']}, 'answerKey': 'A'}


In [10]:
print(len(dataset["train"]))
print(len(dataset["validation"]))
print(len(dataset["test"]))

9741
1221
1140


In [11]:
example = dataset["train"][0]

print("Question:")
print(example["question"])

print("\nChoices:")

for label, text in zip(example["choices"]["label"],
                       example["choices"]["text"]):
    print(f"{label}. {text}")

print("\nCorrect:", example["answerKey"])

Question:
The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?

Choices:
A. ignore
B. enforce
C. authoritarian
D. yell at
E. avoid

Correct: A


In [12]:
# We check if the test labels are hidden or not
print(set(dataset["test"]["answerKey"][:20]))

{''}


In [13]:
# Cell 10: answer distribution — check for class imbalance
import pandas as pd
train_df = dataset["train"].to_pandas()
print(train_df["answerKey"].value_counts())

answerKey
D    1985
B    1973
C    1946
E    1928
A    1909
Name: count, dtype: int64


In [14]:
# Cell 11: question length — tells you if max_seq_length=384 is overkill
train_df["question_len"] = train_df["question"].str.split().str.len()
print(train_df["question_len"].describe())

count    9741.000000
mean       13.249974
std         5.516564
min         3.000000
25%         9.000000
50%        12.000000
75%        16.000000
max        63.000000
Name: question_len, dtype: float64


In [25]:
# Cell A: build config dict directly
config = {
    "wandb_project": "commonsense-qa-mcq",
    "wandb_entity": None,
    "run_name": "tiny-transformer-scratch",
    "vocab_size": 30522,
    "max_seq_length": 96,
    "hidden_size": 256,
    "num_layers": 4,
    "num_heads": 4,
    "ffn_size": 1024,
    "dropout": 0.1,
    "batch_size": 8,
    "grad_accum_steps": 2,
    "mixed_precision": True,
    "epochs": 5,
    "lr": 3e-4,
    "checkpoint_dir": "/content/drive/MyDrive/commonsense-qa/checkpoints",
}

os.makedirs(config["checkpoint_dir"], exist_ok=True)
print(config)

{'wandb_project': 'commonsense-qa-mcq', 'wandb_entity': None, 'run_name': 'tiny-transformer-scratch', 'vocab_size': 30522, 'max_seq_length': 96, 'hidden_size': 256, 'num_layers': 4, 'num_heads': 4, 'ffn_size': 1024, 'dropout': 0.1, 'batch_size': 8, 'grad_accum_steps': 2, 'mixed_precision': True, 'epochs': 5, 'lr': 0.0003, 'checkpoint_dir': '/content/drive/MyDrive/commonsense-qa/checkpoints'}


In [26]:
# Cell B: write it to disk so the script can read it
import os
import yaml

os.makedirs("configs", exist_ok=True)
with open("configs/config.yaml", "w") as f:
    yaml.dump(config, f)

print(open("configs/config.yaml").read())

batch_size: 8
checkpoint_dir: /content/drive/MyDrive/commonsense-qa/checkpoints
dropout: 0.1
epochs: 5
ffn_size: 1024
grad_accum_steps: 2
hidden_size: 256
lr: 0.0003
max_seq_length: 96
mixed_precision: true
num_heads: 4
num_layers: 4
run_name: tiny-transformer-scratch
vocab_size: 30522
wandb_entity: null
wandb_project: commonsense-qa-mcq



# Data Training

In [27]:
import torch
from torch.utils.data import Dataset

OPTION_COLS = ["A", "B", "C", "D", "E"]

class CommonsenseQADataset(Dataset):
    def __init__(self, hf_split, tokenizer, max_seq_length=96):
        self.data = hf_split
        self.tokenizer = tokenizer
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        example = self.data[idx]
        question = example["question"]
        label_to_text = dict(zip(example["choices"]["label"], example["choices"]["text"]))
        option_texts = [label_to_text[col] for col in OPTION_COLS]

        encoded = self.tokenizer(
            [question] * len(OPTION_COLS),
            option_texts,
            padding="max_length",
            truncation=True,
            max_length=self.max_seq_length,
            return_tensors="pt",
        )

        answer_key = example.get("answerKey", "")
        label = OPTION_COLS.index(answer_key) if answer_key in OPTION_COLS else -1

        return {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "label": torch.tensor(label, dtype=torch.long),
        }

In [30]:
!git init
!git remote add origin https://${GITHUB_TOKEN}@github.com/ChinmayeeAwale/dlgenaiproject.git

Reinitialized existing Git repository in /content/dlgenaiproject/.git/
error: remote origin already exists.


In [31]:
!git add .

In [32]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   commonsense_qa_scratch_model.ipynb
	new file:   configs/config.yaml



In [33]:
!git add "/content/drive/MyDrive/Colab Notebooks/Model_1.ipynb"

fatal: /content/drive/MyDrive/Colab Notebooks/Model_1.ipynb: '/content/drive/MyDrive/Colab Notebooks/Model_1.ipynb' is outside repository at '/content/dlgenaiproject'


In [34]:
!git restore --staged commonsense_qa_scratch_model.ipynb

In [35]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   configs/config.yaml

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	commonsense_qa_scratch_model.ipynb

